# Day 4 — ILT 2: Building the Bronze Layer — Design & Strategy
### GlobalMart Data Engineering · 1:30 PM – 3:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- Explain the role and responsibilities of the Bronze layer in medallion architecture
- Apply the 3 Golden Rules of Bronze design
- Define the standard audit columns every Bronze table must have
- Design Bronze table schemas for both of GlobalMart's ingestion pathways
- Choose the correct write mode (append vs overwrite) per pathway
- Describe how CDC events are stored in Bronze — as an event log, not current state

---

## Agenda

| Time | Topic |
|------|-------|
| 1:30 | Bronze in the medallion architecture |
| 1:40 | The 3 Golden Rules of Bronze |
| 1:50 | Audit columns — what, why, which |
| 2:05 | Table naming conventions |
| 2:15 | Write modes — append vs overwrite |
| 2:25 | CDC Bronze design |
| 2:40 | GlobalMart's 6 Bronze table designs |
| 2:55 | What NOT to do in Bronze + Q&A |

---
## 1. Bronze in the Medallion Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                  MEDALLION ARCHITECTURE                       │
│                                                               │
│  Sources → BRONZE → SILVER → GOLD → Consumers                │
│                                                               │
│  BRONZE   = Raw ingestion layer                               │
│             "Land first, ask questions later"                 │
│                                                               │
│  SILVER   = Cleaned, conformed, enriched                      │
│             "Make it trustworthy"                             │
│                                                               │
│  GOLD     = Business-ready aggregates                         │
│             "Make it useful" — this is where fact_sales lives │
└──────────────────────────────────────────────────────────────┘
```

### Bronze Is NOT a Staging Area

Common misconception: Bronze = temporary holding area, delete after Silver is built.

**Wrong.** Bronze is permanent.

| Staging Area | Bronze Layer |
|-------------|-------------|
| Temporary — deleted after processing | Permanent — kept forever |
| No audit trail | Full audit trail (when + from where) |
| No time travel | Delta time travel across all historical loads |
| No schema enforcement | Evolves with the source, tracked by Auto Loader / Lakeflow Connect |
| Can't reprocess | Can always reprocess Silver from Bronze |

### Why Keep Bronze Forever?

> **Bronze is your insurance policy.**
>
> If Silver has a bug (wrong transformation, bad join), you don't go back to the source.
> You reprocess from Bronze — faster, cheaper, and the source may have already changed.

Real example:
```
Silver job has a timezone bug → orders from 11 PM to 1 AM assigned wrong date
Fix: correct the transformation, rerun Silver from Bronze → problem solved
Without Bronze: would need to re-extract from Postgres and re-request ADLS file
                history — painful, and the files may no longer be there
```

---
## 2. The 3 Golden Rules of Bronze

### Rule 1 — Land Raw, Transform Never

```
Bronze DOES:                          Bronze DOES NOT:
  ✅ Append incoming data               ❌ Rename columns
  ✅ Add audit columns                  ❌ Cast data types
  ✅ Store CDC events as-is             ❌ Join with other tables
  ✅ Preserve source column names       ❌ Filter rows
  ✅ Handle schema evolution            ❌ Aggregate
  ✅ Write to Delta                     ❌ Apply business rules
```

The only columns Bronze adds are **audit columns** — everything from the source lands as-is.

---

### Rule 2 — Idempotent Writes

**Idempotent** = running the ingestion multiple times produces the same result. No duplicates.

```
Run 1: products.csv → 12,000 rows in Bronze
Run 2: products.csv → same 12,000 rows (not re-processed)
Run 3: products_v2.csv → 12,040 rows (only 40 new rows added)
```

How we achieve this per pathway:
- **ADLS Autoloader (products, customers, address, payments):** checkpoint — tracks every processed file
- **Postgres CDC (orders, order_items):** WAL offset — Lakeflow Connect tracks the last consumed event via the replication slot

---

### Rule 3 — Never Delete, Only Append (Bronze Is Append-Only)

```
Both GlobalMart pathways are append-only in Bronze:

  ✅ NEVER delete rows from Bronze
  ✅ If Postgres deletes an order → Bronze stores the DELETE *event* (a new row with _cdc_op = 'DELETE')
  ✅ If a corrected products.csv is dropped → Bronze appends the new file's rows alongside the old ones
  ✅ Silver handles MERGE / dedup to reflect the true current state

Neither pathway ever overwrites Bronze wholesale. There is no "full snapshot, overwrite
daily" source in GlobalMart's real pipeline — every table here is naturally append-only.
```

---
## 3. Audit Columns — The Bronze Standard

Every Bronze table must have these audit columns added during ingestion. This matches the real audit-column set used in GlobalMart's actual `gbmart.bronze.*` tables — deliberately minimal, not a kitchen-sink of "just in case" columns:

| Column | Data Type | Value | Purpose | Which Pathway |
|--------|-----------|-------|----------|----------------|
| `_ingested_at` | timestamp | `current_timestamp()` | When the row was written to Bronze | Both |
| `_source_file` | string | `col("_metadata.file_path")` | Which file this row came from | Autoloader only |
| `_cdc_op` | string | INSERT / UPDATE / DELETE | CDC operation type | CDC (Postgres) only |

---

### The Real Pattern

```python
from pyspark.sql.functions import current_timestamp, col

# Autoloader (ADLS file) sources — real pattern from gbmart.bronze.customers:
df = (
    df.withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

# CDC (Postgres) sources — _cdc_op comes from the CDC event itself.
# Lakeflow Connect adds it automatically in production; the Day 2 HOL 2
# manual walkthrough derived it from the WAL text you inspected by hand.
df = (
    df.withColumn("_ingested_at", current_timestamp())
)
```

### Why Keep It This Minimal?

A common instinct is to add `_source_system`, `_batch_id`, `_load_type`, and similar "just in case" columns to every Bronze table. GlobalMart's real build doesn't — and the reason matters:

```
_source_file already answers "which pathway, which file, when" for Autoloader
  (the path itself encodes the source folder — no separate _source_system needed)

_cdc_op already answers "what kind of change" for CDC
  (there's only one CDC source — Postgres — so _source_system would always be
   the same literal value on every row: zero information content)

Every extra column is one more thing to keep correct across every ingestion
script, forever. Add a column only when a real query will filter or group by it.
```

---
## 4. Table Naming Conventions

### Unity Catalog Naming — the real, live pattern

GlobalMart runs on Unity Catalog *today* — this isn't a "later" preview, it's how every table you build from here on is named:

```sql
-- Fully qualified: CATALOG.SCHEMA.TABLE
gbmart.bronze.orders
gbmart.bronze.order_items
gbmart.bronze.products
gbmart.bronze.customers
gbmart.bronze.addresses
gbmart.bronze.payments

gbmart.silver.orders          -- cleaned, conformed
gbmart.gold.fact_sales         -- business-ready

-- Catalog = gbmart          (the whole GlobalMart project)
-- Schema  = bronze/silver/gold   (the medallion layer — this IS the namespace,
--                                  so it never needs repeating in the table name)
-- Table   = the entity, plain and simple (no source-system prefix needed —
--                                          the catalog already tells you that)
```

### Why the Table Name Doesn't Need a Prefix

A common instinct coming from path-based storage (`bronze_supabase_orders`, `bronze_adls_products`) is to encode the layer and source into the table name itself. In Unity Catalog, don't — the three-level namespace already carries that information:

```
❌ Redundant:  gbmart.bronze.bronze_adls_products
✅ Clean:      gbmart.bronze.products

The schema "bronze" already says what layer this is.
The catalog "gbmart" already says what project this is.
The table name just needs to say what the data IS: products.
```

### Reading and Writing with Three-Level Names

```python
# No abfss:// path needed for a Unity Catalog managed table — just the name:
df = spark.table("gbmart.bronze.products")

df.write.format("delta").mode("append").saveAsTable("gbmart.bronze.products")

# Or with streaming:
query.toTable("gbmart.bronze.products")
```

The external location (`gbmart-ext-loc`) is only needed for the **source-side** read (the raw CSVs in `raw-data/`) — once the Bronze table itself is created, it's a normal managed table addressed by name, same as any SQL table you've used before.

---
## 5. Write Modes — Append vs Overwrite

### When to Append (GlobalMart's Default — Every Bronze Table)

Use **append** when the source produces an ordered stream of events or a new batch of rows:

```
Postgres CDC:       INSERT/UPDATE/DELETE events → append each event
ADLS Auto Loader:   new files land → append rows from each file
```

```python
# Batch append:
df.write.format("delta").mode("append").saveAsTable("gbmart.bronze.products")

# Streaming append:
raw_stream.writeStream.format("delta").outputMode("append").toTable("gbmart.bronze.products")
```

### When Overwrite Would Apply (Not Currently Used by GlobalMart)

Overwrite is only correct when a source returns a **full state snapshot with no event log** — for example, a nightly export that always contains 100% of current rows and no history. GlobalMart has no such source today: both Postgres and the ADLS file drops are naturally incremental. Know the pattern anyway — you'll meet full-snapshot sources on other projects.

### The Danger of Overwrite on an Append Source

```
❌ Wrong: overwriting Bronze orders every day
   Day 1: 1,400,000 rows → Bronze has 1,400,000 rows
   Day 2: 1,500,000 rows → Bronze overwritten → 1,500,000 rows
   Day 3: discovers Day 1 data had a bug → CANNOT go back

✅ Correct: appending every day
   Day 1: 1,400,000 rows appended
   Day 2: 100,000 new rows appended
   Day 3: use time travel to go back to the Day 1 state
```

| Table | Write Mode | Reason |
|-------|------------|--------|
| `gbmart.bronze.orders` | Append | Each CDC event is a new record |
| `gbmart.bronze.order_items` | Append | Each CDC event is a new record |
| `gbmart.bronze.products` | Append | Each file drop = new batch of rows |
| `gbmart.bronze.customers` | Append | Each file drop = new batch of rows |
| `gbmart.bronze.addresses` | Append | Each file drop = new batch of rows |
| `gbmart.bronze.payments` | Append | Each file drop = new batch of rows |

---
## 6. CDC Bronze Design

CDC (Change Data Capture) produces an **event log** — not the current state of the table.

### What CDC Events Look Like in Bronze

```
gbmart.bronze.orders (CDC event log):

_cdc_op  | _ingested_at          | order_id   | customer_id | order_date | ...
---------|------------------------|------------|-------------|------------
INSERT   | 2026-06-15 09:00:00   | ORD-001    | CUST-001    | 2026-06-15 |
INSERT   | 2026-06-15 09:00:05   | ORD-002    | CUST-002    | 2026-06-15 |
UPDATE   | 2026-06-15 10:15:00   | ORD-001    | CUST-001    | 2026-06-15 |  ← status changed
INSERT   | 2026-06-15 11:00:00   | ORD-003    | CUST-003    | 2026-06-15 |
DELETE   | 2026-06-15 12:30:00   | ORD-002    | CUST-002    | 2026-06-15 |  ← cancelled
```

### Key Point: Bronze Stores ALL Events

```
Bronze does NOT apply MERGE — it appends EVERY event.

For ORD-001:
  Bronze has 2 rows: INSERT + UPDATE
  (Both rows exist in Bronze)

Silver applies MERGE:
  Takes the LATEST event per order_id → keeps only current state
  gbmart.silver.orders has 1 row for ORD-001 (the UPDATEd version)
```

### Why Store All Events in Bronze?

1. **Audit trail:** You can see every change ever made to any order
2. **Replayability:** Silver can be rebuilt with ANY version of the transformation logic
3. **Debugging:** If Silver shows wrong data, trace back to the exact CDC event
4. **Compliance:** Financial regulations often require full event logs

### The `_cdc_op` Column Values

| `_cdc_op` | Meaning | Silver Action |
|-------|---------|---------------|
| `INSERT` (or `r` for initial snapshot) | New row | INSERT into Silver |
| `UPDATE` | Row was modified | MERGE (update) into Silver |
| `DELETE` | Row was deleted at source | MERGE (delete) or soft-delete flag in Silver |

> Recall Day 2 HOL 2: the raw WAL text used `INSERT`/`UPDATE`/`DELETE` markers extracted via `regexp_extract`. Lakeflow Connect gives you this as a clean structured column automatically — no regex needed in production.

In [ ]:
# ─── ILLUSTRATIVE: Standard Bronze write pattern ──────────────────────────────
# Reference snippet showing the audit column pattern for the CDC pathway.
# In production this is what Lakeflow Connect does automatically; conceptually
# it's a MERGE-free append of every change event straight into Bronze.

"""
from pyspark.sql.functions import current_timestamp, lit

# In production, Lakeflow Connect lands CDC events directly — no notebook code
# needed at all. This snippet shows what the underlying write looks like so the
# mechanics aren't a black box: every change event, appended, with audit columns.

df_bronze = (
    df_cdc_events
    .withColumn("_ingested_at", current_timestamp())
)

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("gbmart.bronze.orders")

print(f"Bronze write complete. Rows: {df_bronze.count():,}")
"""

print("Key patterns in every Bronze write:")
print("  1. withColumn('_ingested_at', current_timestamp())")
print("  2. .mode('append') — every GlobalMart Bronze table is append-only")
print("  3. .option('mergeSchema', 'true') for schema evolution")
print("  4. .saveAsTable('gbmart.bronze.<table>') — Unity Catalog managed table,")
print("     no abfss:// path and no storage key needed in the write itself")

---
## 7. GlobalMart's 6 Bronze Table Designs

### CDC Pathway (Lakeflow Connect) — `gbmart.bronze.*`

```
gbmart.bronze.orders:
  _cdc_op          STRING    ← CDC operation (INSERT/UPDATE/DELETE)
  _ingested_at     TIMESTAMP ← when Bronze wrote this row
  order_id         STRING
  customer_id      STRING
  order_date       STRING    ← kept as STRING in Bronze (Silver casts to DATE)
  shipping_date    STRING
  order_channel    STRING
  ... (all source columns)

gbmart.bronze.order_items:
  _cdc_op, _ingested_at  ← same audit columns
  order_item_id    STRING
  order_id         STRING
  product_id       STRING
  quantity         STRING
  unit_price       STRING
  ... (all source columns)
```

### Autoloader Pathway (ADLS File Drops) — `gbmart.bronze.*`

```
gbmart.bronze.products:
  _ingested_at     TIMESTAMP
  _source_file     STRING    ← full ADLS path of the source CSV
  product_id       STRING
  product_name     STRING
  category         STRING
  actual_price_inr DOUBLE
  ... (all CSV columns)

gbmart.bronze.customers:
  _ingested_at, _source_file  ← same audit columns
  customer_id      STRING
  first_name       STRING
  last_name        STRING
  email            STRING
  ... (all CSV columns — same shape you explored in Day 1 HOL)

gbmart.bronze.addresses / gbmart.bronze.payments: same audit-column pattern,
  entity-specific business columns (address_id/customer_id/city/... and
  payment_id/order_id/payment_method_id/amount/...)
```

---
## 8. What NOT to Do in Bronze

| Anti-Pattern | Why It's Wrong | Correct Approach |
|-------------|---------------|------------------|
| Casting `order_date` STRING → DATE | If casting fails, you lose data | Cast in Silver |
| Renaming `order_channel` to `channel` | Breaks traceability to source | Rename in Silver |
| Filtering cancelled orders | Might need them later for analysis | Filter in Silver |
| Joining `orders` to `customers` | Bronze tables are independent | Join in Silver/Gold |
| Aggregating daily totals | Aggregation belongs in Gold | Aggregate in Gold |
| Deduplicating | Dedup logic may change | Dedup in Silver |
| Dropping columns that seem unused | Nothing is unused until proved otherwise | Keep all columns |

---

## Key Takeaways

1. **Bronze = permanent raw layer** — not a staging area, kept forever
2. **3 Golden Rules:** land raw, idempotent writes, never delete — Bronze is append-only for every GlobalMart table
3. **Audit columns, kept minimal:** `_ingested_at` on every table, plus `_source_file` (Autoloader) or `_cdc_op` (CDC) — no columns without a real downstream use
4. **CDC Bronze = event log** — append ALL events, Silver does the MERGE
5. **Write mode:** append, always — GlobalMart has no full-snapshot source
6. **Never transform in Bronze** — land first, transform in Silver
7. **Naming:** `gbmart.<layer>.<entity>` — the catalog/schema already say project and layer, so the table name is just the entity, nothing more

---

## Discussion Questions

1. *A Postgres DELETE event arrives via Lakeflow Connect. Should Bronze delete the row from the table? Why or why not?*

2. *Your `gbmart.bronze.products` table has 2 million rows. You discover a bug in the ingestion job that corrupted rows from last Tuesday's file. Without a `_batch_id` column, how would you find and fix the affected rows using `_source_file` and `_ingested_at` instead?*

3. *A junior engineer suggests casting all date columns to `DATE` type in Bronze to make Silver simpler. What's the risk?*

4. *Why does GlobalMart never use overwrite mode for any Bronze table?*

5. *What happens to `gbmart.bronze.customers` if next week's `customers.csv` has a new column? What config prevents a pipeline crash?*